# **Start Section:**


In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d05_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # concrete train
    shutil.copy(glob.glob("/kaggle/input/**/d05_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # concrete test
    print("Kaggle: concrete data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: concrete data copied to ./drive/MyDrive/Regression/concrete_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel.
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 bayesian-optimization==3.2.0 properscoring==0.1 XlsxWriter==3.2.9 mapie==0.6.0 puncc==0.8.0
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
import ngboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from sklearn.model_selection import KFold
from PIL import Image as PImage
from openpyxl.utils.dataframe import dataframe_to_rows
from pytorch_tabnet import TabNetRegressor
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


In [4]:
# Go to find & replace button and replace (concrete_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.


In [5]:
feature_names = ['C', 'mp', 'FA', 'CA', 'F', 'W_P', 'Adm']
TARGET_LABEL = 'Compressive strength (str)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)


In [6]:
train_data_path = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")


Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))



Shape of training data: (189, 8)
First 5 rows of training data:
         C    mp     FA      CA       F       W_P    Adm    str
0  280.80  70.2  858.0  1183.0    0.00  0.450000  0.610  21.56
1  372.15   0.0  975.0   525.0   52.85  0.493081  7.000  34.00
2  360.00  75.0  975.0   525.0   40.00  0.509722  7.000  28.00
3  364.30   0.0  975.0   525.0  110.40  0.503706  7.000  42.00
4  315.00  31.5  780.0  1110.0    0.00  0.370000  5.355  25.82

Shape of test data: (95, 8)
First 5 rows of test data:
         C     mp     FA     CA      F       W_P   Adm   str
0  364.30    0.0  975.0  525.0  60.40  0.503706   7.0  34.0
1  344.30   75.0  975.0  525.0  55.40  0.532965   7.0  36.0
2  390.00    0.0  975.0  525.0  60.00  0.470513   7.0  36.0
3  352.15   75.0  975.0  525.0  47.85  0.521085   7.0  35.0
4  400.00  160.0  801.0  801.0  40.00  0.300000  10.3  44.6


In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)



Shape of X_train: (189, 7)
Shape of y_train: (189,)
Shape of X_test: (95, 7)
Shape of y_test: (95,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])



First five rows of normalized X_train:
[[-1.56879184  0.95792014 -0.09685034  1.33138508 -0.85369002  0.16810471
  -0.08799752]
 [ 0.40300047 -0.7708921   0.73537613 -0.93459372  0.061075    0.78285857
  -0.0664621 ]
 [ 0.14074238  1.07612953  0.73537613 -0.93459372 -0.16134185  1.020321
  -0.0664621 ]
 [ 0.233558   -0.7708921   0.73537613 -0.93459372  1.05719093  0.93447436
  -0.0664621 ]
 [-0.83058388  0.00485698 -0.65166799  1.07999229 -0.85369002 -0.97347298
  -0.07200604]]

First five rows of normalized X_test:
[[ 0.233558   -0.7708921   0.73537613 -0.93459372  0.19175572  0.93447436
  -0.0664621 ]
 [-0.19814256  1.07612953  0.73537613 -0.93459372  0.1052122   1.35199213
  -0.0664621 ]
 [ 0.78829322 -0.7708921   0.73537613 -0.93459372  0.18483223  0.4608195
  -0.0664621 ]
 [-0.02870009  1.07612953  0.73537613 -0.93459372 -0.02546852  1.18246784
  -0.0664621 ]
 [ 1.0041435   3.1694207  -0.50229401  0.01587763 -0.16134185 -1.97235346
  -0.05534052]]


# **Functions:**


In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']


In [12]:
# FIX M4: one miscoverage level for the conformal notebooks. AlphaNet LEARNS its own level alpha~,
# so ALPHA is only the nominal reference value reported next to it.
ALPHA = 0.1


# FIX A1: lambda is the size/coverage trade-off chosen by the user (Gauthier, Bach & Jordan 2026,
# eq. 9). The official code reports every lambda separately and never compares losses across
# lambdas. This table evaluates, for EVERY lambda, its best run (lowest final training loss within
# that lambda) on the test set, so the trade-off can be seen.
def _acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test, y_test):
    rows = []
    for lam, res in sorted(all_results.items(), key=lambda kv: kv[0]):
        final_losses = res["all_losses"][:, -1]
        r = int(np.argmin(final_losses))
        ev = evaluate_acp(model=model, alpha_net=res["models"][r], X_calib=X_calib, y_calib=y_calib,
                          X_test=X_test, y_test=y_test)
        a = ev.get("alpha_hat", ev.get("alpha"))
        rows.append({
            "lambda": lam,
            "best_run (1-based)": r + 1,
            "final_train_loss": float(final_losses[r]),
            "alpha_learned": a,
            "Target_coverage": 1.0 - a,
            "test_coverage": ev["coverage"],
            "width": ev["interval_width"],
            "Miss_over_alpha": (1.0 - ev["coverage"]) / a,
        })
    return pd.DataFrame(rows)


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _to_numpy_2d(x):
    arr = x.to_numpy() if hasattr(x, "to_numpy") else np.asarray(x)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    return arr.astype(np.float32)

def _to_numpy_1d(y):
    arr = y.to_numpy() if hasattr(y, "to_numpy") else np.asarray(y)
    return arr.reshape(-1).astype(np.float32)

def _ensure_excel_workbook(excel_file_path):
    if not excel_file_path:
        return
    os.makedirs(os.path.dirname(excel_file_path), exist_ok=True)
    if not os.path.exists(excel_file_path):
        wb = Workbook()
        wb.active.title = "Init"
        wb.save(_ensure_parent_dir(excel_file_path))

def save_plot_to_excel(fig, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    with io.BytesIO() as buf:
        fig.savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        img = Image(PImage.open(buf))
        workbook = load_workbook(_ensure_excel_file(excel_file_path))
        base_name = sheet_name
        i = 1
        while sheet_name in workbook.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        worksheet = workbook.create_sheet(title=sheet_name)
        worksheet.add_image(img, "A1")
        workbook.save(_ensure_parent_dir(excel_file_path))

def save_values_to_excel(results_dict, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    df = pd.DataFrame(results_dict)
    workbook = load_workbook(_ensure_excel_file(excel_file_path))
    base_name = sheet_name
    i = 1
    while sheet_name in workbook.sheetnames:
        sheet_name = f"{base_name}_{i}"
        i += 1
    worksheet = workbook.create_sheet(title=sheet_name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            worksheet.cell(row=r_idx, column=c_idx, value=value)
    workbook.save(_ensure_parent_dir(excel_file_path))

def save_dataframe_to_excel(df, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    workbook = load_workbook(_ensure_excel_file(excel_file_path))
    base_name = sheet_name
    i = 1
    while sheet_name in workbook.sheetnames:
        sheet_name = f"{base_name}_{i}"
        i += 1
    worksheet = workbook.create_sheet(title=sheet_name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            worksheet.cell(row=r_idx, column=c_idx, value=value)
    workbook.save(_ensure_parent_dir(excel_file_path))

class AlphaNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )
        # FIX A2: start the output near alpha~ = 1 (sigmoid(5) ~ 0.993), as in the official code
        # (github.com/GauthierE/adaptive-coverage-policies, initialize_to_one), so alpha~ starts in the
        # region alpha~ > 1/(n+1) where the width formula is valid. Was: random initial output.
        with torch.no_grad():
            self.net[2].weight.data.normal_(0, 0.01)
            self.net[2].bias.data.fill_(5.0)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def interval_size(scores, alpha, eps=1e-6):
    n = scores.numel()
    denominator = torch.clamp(alpha * (n + 1) - 1.0, min=eps)
    return 2.0 * scores.sum() / denominator

def _compute_scores_without_index(base_model, X_calib, y_calib, holdout_idx):
    loo_X = np.delete(X_calib, holdout_idx, axis=0)
    loo_y = np.delete(y_calib, holdout_idx, axis=0)
    return np.abs(base_model.predict(loo_X) - loo_y)

def _build_loo_feature_matrix(base_model, X_calib, y_calib):
    rows = []
    for idx in range(len(X_calib)):
        scores = _compute_scores_without_index(base_model, X_calib, y_calib, idx)
        rows.append([scores.sum()])
    return torch.tensor(np.asarray(rows, dtype=np.float32))

def _resolve_model_params(model_class, best_params):
    params = dict(best_params or {})
    if "TabNet" in model_class.__name__ and "verbose" in params:
        params.pop("verbose", None)
    try:
        model_class(**params)
        return params
    except Exception as e:
        # FIX B5: was a silent fallback to default parameters
        print(f"[Warning] {model_class.__name__}: tuned parameters rejected ({type(e).__name__}: {e}); "
              "falling back to the model's DEFAULT parameters")
        return {}

def _train_alpha_net(
    base_model,
    X_calib,
    y_calib,
    lambdas=(10, 20, 50),
    num_runs=3,
    epochs=50,
    batch_size=32,
    learning_rate=1e-2,  # FIX A3 (was 1e-3)
    min_steps=2000,  # FIX A3
    alpha_clip=1e-3,
    seed=42,
):
    X_train_alpha = _build_loo_feature_matrix(base_model, X_calib, y_calib)
    # FIX A2: the network input (sum of leave-one-out errors) is divided by its mean, so inputs are ~1 on
    # every dataset (raw sums can be in the thousands, where the sigmoid is flat and training stalls).
    # The same number divides the test feature in evaluate_acp (stored on each net as feature_scale_).
    feature_scale = float(X_train_alpha.mean())
    feature_scale = feature_scale if feature_scale > 0 else 1.0
    X_train_alpha = X_train_alpha / feature_scale
    # FIX A3: leave-one-out score vectors computed once (identical values; was one predict() per row per step)
    loo_scores = [torch.tensor(_compute_scores_without_index(base_model, X_calib, y_calib, i), dtype=torch.float32, device=DEVICE)
                  for i in range(len(X_calib))]
    all_results = {}
    for lambda_reg in lambdas:
        lambda_losses = []
        lambda_sizes = []
        lambda_alphas = []
        lambda_models = []
        for run_idx in range(num_runs):
            local_seed = seed + run_idx
            torch.manual_seed(local_seed)
            np.random.seed(local_seed)
            random.seed(local_seed)
            dataset = TensorDataset(X_train_alpha, torch.arange(len(X_train_alpha), dtype=torch.long))
            loader = DataLoader(dataset, batch_size=min(batch_size, len(X_train_alpha)), shuffle=True)
            alpha_net = AlphaNet(input_dim=X_train_alpha.shape[1]).to(DEVICE)
            optimizer = optim.Adam(alpha_net.parameters(), lr=learning_rate)
            all_losses = []
            all_sizes = []
            all_alphas = []
            # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
            # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
            # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
            # now matches the exact minimiser of the same loss for every lambda.
            n_epochs = max(epochs, int(np.ceil(min_steps / len(loader))))
            for _ in range(n_epochs):
                epoch_losses = []
                epoch_sizes = []
                epoch_alphas = []
                alpha_net.train()
                for x_batch, idx_batch in loader:
                    x_batch = x_batch.to(DEVICE)
                    alpha_pred = torch.clamp(alpha_net(x_batch), min=alpha_clip, max=1.0 - alpha_clip)
                    batch_sizes = []
                    for j, idx in enumerate(idx_batch.tolist()):
                        batch_sizes.append(interval_size(loo_scores[idx], alpha_pred[j]))  # FIX A3: cached
                    batch_sizes = torch.stack(batch_sizes)
                    loss = (batch_sizes + lambda_reg * alpha_pred).mean()
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    epoch_losses.append(float(loss.item()))
                    epoch_sizes.append(float(batch_sizes.mean().item()))
                    epoch_alphas.append(float(alpha_pred.mean().item()))
                all_losses.append(float(np.mean(epoch_losses)))
                all_sizes.append(float(np.mean(epoch_sizes)))
                all_alphas.append(float(np.mean(epoch_alphas)))
            alpha_net.eval()
            lambda_losses.append(np.asarray(all_losses, dtype=np.float32))
            lambda_sizes.append(np.asarray(all_sizes, dtype=np.float32))
            lambda_alphas.append(np.asarray(all_alphas, dtype=np.float32))
            alpha_net.feature_scale_ = feature_scale  # FIX A2
            lambda_models.append(alpha_net)
        all_results[lambda_reg] = {
            "all_losses": np.vstack(lambda_losses),
            "all_sizes": np.vstack(lambda_sizes),
            "all_alphas": np.vstack(lambda_alphas),
            "models": lambda_models,
        }
    return all_results

def _moving_average(values, window=10):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    if window <= 1 or arr.size < window:
        return arr.copy()
    kernel = np.ones(window, dtype=float) / float(window)
    return np.convolve(arr, kernel, mode="valid")

def _smooth_runs(run_matrix, window=10):
    smoothed_runs = [_moving_average(run_curve, window=window) for run_curve in np.asarray(run_matrix)]
    if not smoothed_runs:
        return np.array([]), np.array([])
    min_len = min(len(curve) for curve in smoothed_runs)
    if min_len == 0:
        return np.array([]), np.array([])
    stacked = np.asarray([curve[:min_len] for curve in smoothed_runs], dtype=float)
    return stacked.mean(axis=0), stacked.std(axis=0)

def plot_acp_training_dynamics(all_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00"]
    for idx, (lambda_reg, result_dict) in enumerate(sorted(all_results.items(), key=lambda item: item[0])):
        color = colors[idx % len(colors)]
        loss_mean, loss_std = _smooth_runs(result_dict["all_losses"], window=10)
        size_mean, size_std = _smooth_runs(result_dict["all_sizes"], window=10)
        alpha_mean, alpha_std = _smooth_runs(result_dict["all_alphas"], window=10)
        if loss_mean.size == 0:
            continue
        epochs_axis = np.arange(1, len(loss_mean) + 1)
        axes[0].plot(epochs_axis, loss_mean, color=color, label=f"lambda={lambda_reg}")
        axes[0].fill_between(epochs_axis, loss_mean - loss_std, loss_mean + loss_std, color=color, alpha=0.2)
        axes[1].plot(epochs_axis, size_mean, color=color, label=f"lambda={lambda_reg}")
        axes[1].fill_between(epochs_axis, size_mean - size_std, size_mean + size_std, color=color, alpha=0.2)
        axes[2].plot(epochs_axis, alpha_mean, color=color, label=f"lambda={lambda_reg}")
        axes[2].fill_between(epochs_axis, alpha_mean - alpha_std, alpha_mean + alpha_std, color=color, alpha=0.2)
    axes[0].set_title("Smoothed Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[1].set_title("Smoothed Mean Size")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Mean Size")
    axes[2].set_title(r"Smoothed Mean $\tilde{\alpha}$")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel(r"Mean $\tilde{\alpha}$")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(loc="best", fontsize="small")
    plt.tight_layout()
    return fig

def _select_best_alpha_net(all_results, lambda_main=50):
    # FIX A1: lambda is the user's size/coverage trade-off; the best RUN is chosen within lambda_main by its
    # final training loss. Was: final losses compared ACROSS lambdas; since the loss contains lambda * alpha~,
    # the smallest lambda (10, the narrowest / lowest-coverage setting) always won.
    if lambda_main not in all_results:
        raise ValueError(f"lambda_main={lambda_main} not in trained lambdas {sorted(all_results)}")
    final_losses = all_results[lambda_main]["all_losses"][:, -1]
    run_idx = int(np.argmin(final_losses))
    return lambda_main, run_idx, all_results[lambda_main]["models"][run_idx]


def evaluate_acp(model, alpha_net, X_calib, y_calib, X_test, y_test):
    X_calib_np = _to_numpy_2d(X_calib)
    y_calib_np = _to_numpy_1d(y_calib)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)
    calib_scores = np.abs(model.predict(X_calib_np) - y_calib_np)
    test_feature = torch.tensor([[calib_scores.sum() / getattr(alpha_net, "feature_scale_", 1.0)]], dtype=torch.float32, device=DEVICE)  # FIX A2
    with torch.no_grad():
        alpha_hat = float(torch.clamp(alpha_net(test_feature), min=1e-3, max=1.0 - 1e-3).item())
    score_tensor = torch.tensor(calib_scores, dtype=torch.float32, device=DEVICE)
    interval_width = float(interval_size(score_tensor, torch.tensor(alpha_hat, device=DEVICE)).item())
    y_pred = np.asarray(model.predict(X_test_np)).reshape(-1)
    lower = y_pred - 0.5 * interval_width
    upper = y_pred + 0.5 * interval_width
    coverage = float(np.mean((y_test_np >= lower) & (y_test_np <= upper)))
    rmse = float(np.sqrt(mean_squared_error(y_test_np, y_pred)))
    mae = float(np.mean(np.abs(y_test_np - y_pred)))
    return {
        "X_test": X_test_np,
        "y_test": y_test_np,
        "y_pred": y_pred,
        "lower": lower,
        "upper": upper,
        "alpha": alpha_hat,
        "interval_width": interval_width,
        "coverage": coverage,
        "rmse": rmse,
        "mae": mae,
        # FIX A1: the guarantee is about the LEARNED level (eq. 3: E[miss / alpha~] <= 1), so report
        # coverage against 1 - alpha~ and the miss rate divided by alpha~.
        "target_coverage": 1.0 - alpha_hat,
        "miss_over_alpha": (1.0 - coverage) / alpha_hat,
    }

def conformal_predictions_ACP(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path,
    alpha=ALPHA,  # FIX M4: was 0.1 (nominal only; AlphaNet learns alpha~)
    lambdas=(10, 20, 50),
    lambda_main=50,  # FIX A1: the lambda used for the main result (paper reports lambda = 50)
    num_runs=3,
    epochs=50,
):
    X_train_np = _to_numpy_2d(X_train)
    y_train_np = _to_numpy_1d(y_train)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)
    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train_np,
        y_train_np,
        test_size=0.35,
        random_state=42,
    )
    safe_params = _resolve_model_params(model_class, best_params)
    model = model_class(**safe_params)
    model.fit(X_fit, y_fit)
    all_results = _train_alpha_net(
        base_model=model,
        X_calib=X_calib,
        y_calib=y_calib,
        lambdas=lambdas,
        num_runs=num_runs,
        epochs=epochs,
    )
    best_lambda, best_run_idx, alpha_net = _select_best_alpha_net(all_results, lambda_main)  # FIX A1
    eval_bundle = evaluate_acp(
        model=model,
        alpha_net=alpha_net,
        X_calib=X_calib,
        y_calib=y_calib,
        X_test=X_test_np,
        y_test=y_test_np,
    )
    y_true = eval_bundle["y_test"]
    y_pred = eval_bundle["y_pred"]
    lower = eval_bundle["lower"]
    upper = eval_bundle["upper"]
    print(f"{model_name} ACP best lambda: {best_lambda}")
    print(f"{model_name} ACP best run: {best_run_idx + 1}")
    print(f"{model_name} ACP alpha: {eval_bundle['alpha']:.4f}")
    print(f"{model_name} ACP width: {eval_bundle['interval_width']:.4f}")
    print(f"{model_name} ACP coverage: {eval_bundle['coverage'] * 100:.2f}%")
    print(f"{model_name} ACP RMSE: {eval_bundle['rmse']:.4f}")
    print(f"{model_name} ACP MAE: {eval_bundle['mae']:.4f}")
    print(f"{model_name} ACP target coverage (1 - alpha~): {eval_bundle['target_coverage'] * 100:.2f}%  |  miss / alpha~: {eval_bundle['miss_over_alpha']:.3f}  # FIX A1")
    fig_pred, axs = plt.subplots(2, 1, figsize=(11, 12))
    idx = np.arange(len(y_true))
    axs[0].scatter(idx, y_true, label="True", color="blue", s=12, alpha=0.7)
    axs[0].fill_between(idx, lower, upper, color="gray", alpha=0.45, label="ACP Interval")
    axs[0].plot(idx, y_pred, color="red", linewidth=1.2, label="Prediction")
    axs[0].set_title(f"ACP Intervals - {model_name}")
    axs[0].set_xlabel("Sample Number")
    axs[0].set_ylabel(f"{TARGET_LABEL}")
    axs[0].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
    abs_err = np.abs(y_true - y_pred)
    widths = upper - lower
    axs[1].scatter(widths, abs_err, s=14, alpha=0.7, color="teal", label="Samples")
    axs[1].axvline(np.mean(widths), linestyle="--", color="gray", label="Mean Width")
    axs[1].axhline(np.mean(abs_err), linestyle="--", color="orange", label="Mean Absolute Error")
    axs[1].set_title(f"ACP Width vs Absolute Error - {model_name}")
    axs[1].set_xlabel("Interval Width")
    axs[1].set_ylabel("Absolute Error")
    axs[1].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
    plt.tight_layout()
    fig_dynamics = plot_acp_training_dynamics(all_results)
    if excel_file_path:
        save_plot_to_excel(fig_pred, excel_file_path, "conformal_predictions_ACP")
        save_plot_to_excel(fig_dynamics, excel_file_path, "acp_training_dynamics")
        save_values_to_excel({
                "ACP_true": y_true,
                "ACP_pred": y_pred,
                "ACP_lower": lower,
                "ACP_upper": upper,
                "ACP_alpha": np.full(len(y_true), eval_bundle["alpha"]),
                "ACP_interval_width": np.full(len(y_true), eval_bundle["interval_width"]),
                "ACP_is_covered": ((y_true >= lower) & (y_true <= upper)).astype(int),
                "ACP_best_lambda": np.full(len(y_true), best_lambda),
                "ACP_best_run": np.full(len(y_true), best_run_idx + 1),
                "ACP_target_coverage": np.full(len(y_true), eval_bundle["target_coverage"]),  # FIX A1
                "ACP_miss_over_alpha": np.full(len(y_true), eval_bundle["miss_over_alpha"]),  # FIX A1
            },
            excel_file_path,
            "conformal_predictions_ACP_values",
        )
        training_summary_rows = []
        training_curve_rows = []
        for lambda_reg, result_dict in sorted(all_results.items(), key=lambda item: item[0]):
            final_losses = result_dict["all_losses"][:, -1]
            final_sizes = result_dict["all_sizes"][:, -1]
            final_alphas = result_dict["all_alphas"][:, -1]
            training_summary_rows.append(
                {
                    "lambda": lambda_reg,
                    "runs": result_dict["all_losses"].shape[0],
                    "best_run": int(np.argmin(final_losses)) + 1,
                    "final_loss_mean": float(np.mean(final_losses)),
                    "final_loss_std": float(np.std(final_losses)),
                    "final_size_mean": float(np.mean(final_sizes)),
                    "final_size_std": float(np.std(final_sizes)),
                    "final_alpha_mean": float(np.mean(final_alphas)),
                    "final_alpha_std": float(np.std(final_alphas)),
                }
            )
            mean_loss_curve = result_dict["all_losses"].mean(axis=0)
            mean_size_curve = result_dict["all_sizes"].mean(axis=0)
            mean_alpha_curve = result_dict["all_alphas"].mean(axis=0)
            for epoch_idx in range(len(mean_loss_curve)):
                training_curve_rows.append(
                    {
                        "lambda": lambda_reg,
                        "epoch": epoch_idx + 1,
                        "mean_loss": float(mean_loss_curve[epoch_idx]),
                        "mean_size": float(mean_size_curve[epoch_idx]),
                        "mean_alpha": float(mean_alpha_curve[epoch_idx]),
                    }
                )
        save_dataframe_to_excel(pd.DataFrame(training_summary_rows), excel_file_path, "acp_training_summary")
        save_dataframe_to_excel(pd.DataFrame(training_curve_rows), excel_file_path, "acp_training_curves")
        save_dataframe_to_excel(_acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test_np, y_test_np),
                                excel_file_path, "acp_lambda_comparison")  # FIX A1
    plt.close(fig_pred)
    plt.close(fig_dynamics)
    eval_bundle["model"] = model
    eval_bundle["alpha_net"] = alpha_net
    eval_bundle["all_results"] = all_results
    eval_bundle["best_lambda"] = best_lambda
    eval_bundle["best_run"] = best_run_idx + 1
    return eval_bundle


In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

def _train_alpha_net_multi(
    base_model,
    X_calib,
    y_calib,
    X_train_alpha,
    lambdas=(10, 20, 50),
    num_runs=3,
    epochs=50,
    batch_size=32,
    lr=1e-2,  # FIX A3 (was 1e-3)
    alpha_clip=1e-3,
    min_steps=2000,  # FIX A3
    seed=42,
):
    # FIX A2: the network input (sum of leave-one-out errors) is divided by its mean, so inputs are ~1 on
    # every dataset (raw sums can be in the thousands, where the sigmoid is flat and training stalls).
    # The same number divides the test feature in evaluate_acp (stored on each net as feature_scale_).
    feature_scale = float(X_train_alpha.mean())
    feature_scale = feature_scale if feature_scale > 0 else 1.0
    X_train_alpha = X_train_alpha / feature_scale
    # FIX A3: leave-one-out score vectors computed once (identical values; was one predict() per row per step)
    loo_scores = [torch.tensor(_compute_scores_without_index(base_model, X_calib, y_calib, i), dtype=torch.float32, device=DEVICE)
                  for i in range(len(X_calib))]
    Dataset = TensorDataset(X_train_alpha, torch.arange(len(X_train_alpha), dtype=torch.long))
    loader = DataLoader(Dataset, batch_size=min(batch_size, len(X_train_alpha)), shuffle=True)

    all_results = {}
    for lambda_reg in lambdas:
        lambda_losses = []
        lambda_sizes = []
        lambda_alphas = []
        lambda_models = []

        for run_idx in range(num_runs):
            torch.manual_seed(seed + run_idx)
            np.random.seed(seed + run_idx)

            alpha_net = AlphaNet(input_dim=X_train_alpha.shape[1]).to(DEVICE)
            optimizer = optim.Adam(alpha_net.parameters(), lr=lr)

            run_losses = []
            run_sizes = []
            run_alphas = []

            # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
            # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
            # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
            # now matches the exact minimiser of the same loss for every lambda.
            n_epochs = max(epochs, int(np.ceil(min_steps / len(loader))))
            for _ in range(n_epochs):
                alpha_net.train()
                epoch_losses = []
                epoch_sizes = []
                epoch_alphas = []

                for x_batch, idx_batch in loader:
                    x_batch = x_batch.to(DEVICE)
                    alpha_pred = torch.clamp(alpha_net(x_batch), min=alpha_clip, max=1.0 - alpha_clip)

                    batch_sizes = []
                    for j, idx in enumerate(idx_batch.tolist()):
                        batch_sizes.append(interval_size(loo_scores[idx], alpha_pred[j]))  # FIX A3: cached

                    batch_sizes = torch.stack(batch_sizes)
                    loss = (batch_sizes + float(lambda_reg) * alpha_pred).mean()

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    epoch_losses.append(float(loss.item()))
                    epoch_sizes.append(float(batch_sizes.mean().item()))
                    epoch_alphas.append(float(alpha_pred.mean().item()))

                run_losses.append(float(np.mean(epoch_losses)))
                run_sizes.append(float(np.mean(epoch_sizes)))
                run_alphas.append(float(np.mean(epoch_alphas)))

            alpha_net.eval()
            lambda_losses.append(run_losses)
            lambda_sizes.append(run_sizes)
            lambda_alphas.append(run_alphas)
            alpha_net.feature_scale_ = feature_scale  # FIX A2
            lambda_models.append(alpha_net)

        all_results[float(lambda_reg)] = {
            "all_losses": np.asarray(lambda_losses, dtype=np.float32),
            "all_sizes": np.asarray(lambda_sizes, dtype=np.float32),
            "all_alphas": np.asarray(lambda_alphas, dtype=np.float32),
            "models": lambda_models,
        }

    return all_results


def _moving_average(values, window=10):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values
    window = max(1, min(int(window), values.size))
    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(values, kernel, mode="valid")


def _smooth_runs(run_matrix, window=10):
    run_matrix = np.asarray(run_matrix, dtype=np.float32)
    if run_matrix.ndim == 1:
        run_matrix = run_matrix.reshape(1, -1)
    if run_matrix.size == 0:
        return np.asarray([]), np.asarray([])

    smoothed_runs = [_moving_average(run, window=window) for run in run_matrix]
    min_len = min(len(curve) for curve in smoothed_runs)
    if min_len == 0:
        return np.asarray([]), np.asarray([])

    trimmed = np.vstack([curve[:min_len] for curve in smoothed_runs])
    return trimmed.mean(axis=0), trimmed.std(axis=0)


def plot_acp_training_dynamics(all_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if not all_results:
        for ax in axes:
            ax.text(0.5, 0.5, "No training results", ha="center", va="center")
            ax.axis("off")
        return fig

    sorted_results = sorted(all_results.items(), key=lambda item: item[0])
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(sorted_results)))

    for color, (lambda_reg, result) in zip(colors, sorted_results):
        loss_mean, loss_std = _smooth_runs(result["all_losses"], window=10)
        size_mean, size_std = _smooth_runs(result["all_sizes"], window=10)
        alpha_mean, alpha_std = _smooth_runs(result["all_alphas"], window=10)

        if loss_mean.size:
            x_loss = np.arange(1, loss_mean.size + 1)
            axes[0].plot(x_loss, loss_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[0].fill_between(x_loss, loss_mean - loss_std, loss_mean + loss_std, color=color, alpha=0.2)

        if size_mean.size:
            x_size = np.arange(1, size_mean.size + 1)
            axes[1].plot(x_size, size_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[1].fill_between(x_size, size_mean - size_std, size_mean + size_std, color=color, alpha=0.2)

        if alpha_mean.size:
            x_alpha = np.arange(1, alpha_mean.size + 1)
            axes[2].plot(x_alpha, alpha_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[2].fill_between(x_alpha, alpha_mean - alpha_std, alpha_mean + alpha_std, color=color, alpha=0.2)

    axes[0].set_title("Smoothed Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    axes[1].set_title("Smoothed Mean Size")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Size")

    axes[2].set_title("Smoothed Mean Alpha-tilde")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Alpha-tilde")

    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize="small")

    plt.tight_layout()
    return fig


def _select_best_alpha_net(all_results, lambda_main=50):
    # FIX A1: lambda is the user's size/coverage trade-off; the best RUN is chosen within lambda_main by its
    # final training loss. Was: final losses compared ACROSS lambdas; since the loss contains lambda * alpha~,
    # the smallest lambda (10, the narrowest / lowest-coverage setting) always won.
    if lambda_main not in all_results:
        raise ValueError(f"lambda_main={lambda_main} not in trained lambdas {sorted(all_results)}")
    losses = all_results[lambda_main]["all_losses"]
    if losses.size == 0:
        raise RuntimeError("AlphaNet training produced no valid runs.")
    final_losses = losses[:, -1]
    run_idx = int(np.argmin(final_losses))
    return {
        "lambda": lambda_main,
        "run_idx": run_idx,
        "final_loss": float(final_losses[run_idx]),
        "model": all_results[lambda_main]["models"][run_idx],
    }


def evaluate_acp(model, alpha_net, X_calib, y_calib, X_test, y_test):
    X_calib_np = _to_numpy_2d(X_calib)
    y_calib_np = _to_numpy_1d(y_calib)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)

    calib_preds = np.asarray(model.predict(X_calib_np)).reshape(-1)
    calib_scores = np.abs(calib_preds - y_calib_np)

    feature = torch.tensor([[calib_scores.sum() / getattr(alpha_net, "feature_scale_", 1.0)]], dtype=torch.float32, device=DEVICE)  # FIX A2
    with torch.no_grad():
        alpha_hat = float(torch.clamp(alpha_net(feature), min=1e-3, max=1.0 - 1e-3).item())

    scores_tensor = torch.tensor(calib_scores, dtype=torch.float32, device=DEVICE)
    interval_width = float(
        interval_size(scores_tensor, torch.tensor(alpha_hat, dtype=torch.float32, device=DEVICE)).item()
    )

    y_pred = np.asarray(model.predict(X_test_np)).reshape(-1)
    lower = y_pred - 0.5 * interval_width
    upper = y_pred + 0.5 * interval_width

    coverage = float(np.mean((y_test_np >= lower) & (y_test_np <= upper)))
    rmse = float(np.sqrt(mean_squared_error(y_test_np, y_pred)))
    mae = float(np.mean(np.abs(y_test_np - y_pred)))

    return {
        "coverage": coverage,
        "interval_width": interval_width,
        "alpha_hat": alpha_hat,
        "rmse": rmse,
        "mae": mae,
        "y_pred": y_pred,
        "lower": lower,
        "upper": upper,
        "y_test": y_test_np,
        # FIX A1: the guarantee is about the LEARNED level (eq. 3: E[miss / alpha~] <= 1), so report
        # coverage against 1 - alpha~ and the miss rate divided by alpha~.
        "target_coverage": 1.0 - alpha_hat,
        "miss_over_alpha": (1.0 - coverage) / alpha_hat,
    }


def run_full_acp_pipeline(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path,
    lambdas=(10, 20, 50),
    lambda_main=50,  # FIX A1: the lambda used for the main result (paper reports lambda = 50)
    num_runs=3,
    epochs=50,
):
    X_train_np = _to_numpy_2d(X_train)
    y_train_np = _to_numpy_1d(y_train)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)

    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train_np, y_train_np, test_size=0.35, random_state=42
    )

    safe_params = _resolve_model_params(model_class, best_params)
    model = model_class(**safe_params)
    model.fit(X_fit, y_fit)

    X_train_alpha = _build_loo_feature_matrix(model, X_calib, y_calib)
    all_results = _train_alpha_net_multi(
        base_model=model,
        X_calib=X_calib,
        y_calib=y_calib,
        X_train_alpha=X_train_alpha,
        lambdas=lambdas,
        num_runs=num_runs,
        epochs=epochs,
    )

    dynamics_fig = plot_acp_training_dynamics(all_results)
    dynamics_fig.suptitle(f"ACP Training Dynamics - {model_name}")
    dynamics_fig.tight_layout(rect=[0, 0, 1, 0.95])
    if excel_file_path:
        save_plot_to_excel(dynamics_fig, excel_file_path, "acp_training_dynamics")
    plt.close(dynamics_fig)

    best_choice = _select_best_alpha_net(all_results, lambda_main)  # FIX A1
    metrics = evaluate_acp(
        model=model,
        alpha_net=best_choice["model"],
        X_calib=X_calib,
        y_calib=y_calib,
        X_test=X_test_np,
        y_test=y_test_np,
    )

    y_true = metrics["y_test"]
    y_pred = metrics["y_pred"]
    lower = metrics["lower"]
    upper = metrics["upper"]

    print(f"{model_name} best lambda: {best_choice['lambda']}")
    print(f"{model_name} best run: {best_choice['run_idx']}")
    print(f"{model_name} ACP alpha_hat: {metrics['alpha_hat']:.4f}")
    print(f"{model_name} ACP width: {metrics['interval_width']:.4f}")
    print(f"{model_name} ACP coverage: {metrics['coverage'] * 100:.2f}%")
    print(f"{model_name} ACP RMSE: {metrics['rmse']:.4f}")
    print(f"{model_name} ACP MAE: {metrics['mae']:.4f}")
    print(f"{model_name} ACP target coverage (1 - alpha~): {metrics['target_coverage'] * 100:.2f}%  |  miss / alpha~: {metrics['miss_over_alpha']:.3f}  # FIX A1")

    fig, axs = plt.subplots(2, 1, figsize=(11, 12))
    idx = np.arange(len(y_true))

    axs[0].scatter(idx, y_true, label="True", color="blue", s=12, alpha=0.7)
    axs[0].fill_between(idx, lower, upper, color="gray", alpha=0.45, label="ACP Interval")
    axs[0].plot(idx, y_pred, color="red", linewidth=1.2, label="Prediction")
    axs[0].set_title(f"ACP Intervals - {model_name}")
    axs[0].set_xlabel("Sample Number")
    axs[0].set_ylabel(f"{TARGET_LABEL}")
    axs[0].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)

    abs_err = np.abs(y_true - y_pred)
    widths = upper - lower
    axs[1].scatter(widths, abs_err, s=14, alpha=0.7, color="teal", label="Samples")
    axs[1].axvline(np.mean(widths), linestyle="--", color="gray", label="Mean Width")
    axs[1].axhline(np.mean(abs_err), linestyle="--", color="orange", label="Mean Absolute Error")
    axs[1].set_title(f"ACP Width vs Absolute Error - {model_name}")
    axs[1].set_xlabel("Interval Width")
    axs[1].set_ylabel("Absolute Error")
    axs[1].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)

    plt.tight_layout()

    if excel_file_path:
        save_plot_to_excel(fig, excel_file_path, "conformal_predictions_ACP")

        save_values_to_excel({
                "ACP_true": y_true,
                "ACP_pred": y_pred,
                "ACP_lower": lower,
                "ACP_upper": upper,
                "ACP_alpha": np.full(len(y_true), metrics["alpha_hat"]),
                "ACP_interval_width": np.full(len(y_true), metrics["interval_width"]),
                "ACP_is_covered": ((y_true >= lower) & (y_true <= upper)).astype(int),
                "ACP_target_coverage": np.full(len(y_true), metrics["target_coverage"]),  # FIX A1
                "ACP_miss_over_alpha": np.full(len(y_true), metrics["miss_over_alpha"]),  # FIX A1
            },
            excel_file_path,
            "conformal_predictions_ACP_values",
        )

        save_values_to_excel({
                "model_name": [model_name],
                "best_lambda": [best_choice["lambda"]],
                "best_run": [best_choice["run_idx"]],
                "best_final_loss": [best_choice["final_loss"]],
                "coverage": [metrics["coverage"]],
                "interval_width": [metrics["interval_width"]],
                "rmse": [metrics["rmse"]],
                "mae": [metrics["mae"]],
                "alpha_hat": [metrics["alpha_hat"]],
                "Target_coverage": [metrics["target_coverage"]],  # FIX A1
                "Miss_over_alpha": [metrics["miss_over_alpha"]],  # FIX A1
            },
            excel_file_path,
            "acp_final_metrics",
        )

        summary = {
            "lambda": [],
            "mean_final_loss": [],
            "std_final_loss": [],
            "mean_final_size": [],
            "std_final_size": [],
            "mean_final_alpha": [],
            "std_final_alpha": [],
        }
        curve_data = {
            "lambda": [],
            "run": [],
            "epoch": [],
            "loss": [],
            "size": [],
            "alpha_tilde": [],
        }

        for lambda_reg, result in sorted(all_results.items(), key=lambda item: item[0]):
            losses = result["all_losses"]
            sizes = result["all_sizes"]
            alphas = result["all_alphas"]

            summary["lambda"].append(lambda_reg)
            summary["mean_final_loss"].append(float(losses[:, -1].mean()))
            summary["std_final_loss"].append(float(losses[:, -1].std()))
            summary["mean_final_size"].append(float(sizes[:, -1].mean()))
            summary["std_final_size"].append(float(sizes[:, -1].std()))
            summary["mean_final_alpha"].append(float(alphas[:, -1].mean()))
            summary["std_final_alpha"].append(float(alphas[:, -1].std()))

            n_runs, n_epochs = losses.shape
            for run_idx in range(n_runs):
                for epoch_idx in range(n_epochs):
                    curve_data["lambda"].append(lambda_reg)
                    curve_data["run"].append(run_idx)
                    curve_data["epoch"].append(epoch_idx + 1)
                    curve_data["loss"].append(float(losses[run_idx, epoch_idx]))
                    curve_data["size"].append(float(sizes[run_idx, epoch_idx]))
                    curve_data["alpha_tilde"].append(float(alphas[run_idx, epoch_idx]))

        save_values_to_excel(summary, excel_file_path, "acp_training_summary")
        save_values_to_excel(curve_data, excel_file_path, "acp_training_curves")
        lambda_df = _acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test_np, y_test_np)  # FIX A1
        print(lambda_df.to_string(index=False))
        save_dataframe_to_excel(lambda_df, excel_file_path, "acp_lambda_comparison")

    plt.close(fig)

    return {
        "model": model,
        "all_results": all_results,
        "best_choice": best_choice,
        "metrics": metrics,
    }


In [15]:
# Unified ACP execution entry point:
# run_full_acp_pipeline(model_class, best_params, X_train, y_train, X_test, y_test, model_name, excel_file_path)


# **Tuned Parameters**


In [16]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/concrete_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/concrete-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


# **Conformal Predictions with Lightgbm**


In [17]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')


In [18]:
run_full_acp_pipeline(
    LGBMRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "LightGBM",
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/LightGBM.xlsx",
)


LightGBM best lambda: 50
LightGBM best run: 1
LightGBM ACP alpha_hat: 0.3433
LightGBM ACP width: 16.9490
LightGBM ACP coverage: 97.89%
LightGBM ACP RMSE: 3.4395
LightGBM ACP MAE: 2.5080
LightGBM ACP target coverage (1 - alpha~): 65.67%  |  miss / alpha~: 0.061  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         15.046669       0.760602         0.239398       0.778947  7.467124         0.290629
   20.0                   2         21.371731       0.534711         0.465289       0.863158 10.710861         0.255918
   50.0                   2         34.081299       0.343321         0.656679       0.978947 16.948996         0.061321


{'model': LGBMRegressor(bagging_freq=0, colsample_bytree=0.5, learning_rate=0.15,
               max_depth=3, min_child_samples=1, min_child_weight=0.01,
               n_estimators=400, n_jobs=-1, num_leaves=63,
               objective='regression', random_state=42, reg_alpha=0.01,
               reg_lambda=10, subsample=0.5, verbose=-1),
 'all_results': {10.0: {'all_losses': array([[15.626344, 15.609658, 15.569247, ..., 15.073075, 15.061397,
           15.0793  ],
          [15.561189, 15.619028, 15.568255, ..., 15.010252, 15.087655,
           15.046669],
          [15.623726, 15.545624, 15.58456 , ..., 15.072179, 15.096566,
           15.09121 ]], dtype=float32),
   'all_sizes': array([[5.7006545, 5.7363067, 5.7960024, ..., 7.450124 , 7.441385 ,
           7.491363 ],
          [5.634736 , 5.7246604, 5.726778 , ..., 7.376224 , 7.5465627,
           7.3676486],
          [5.70128  , 5.6617117, 5.7708144, ..., 7.515838 , 7.4446416,
           7.4786267]], dtype=float32),
   'all_alp

# **Conformal Predictions with XGBoost**


In [19]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')


In [20]:
run_full_acp_pipeline(
    XGBRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "XGBoost",
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/XGBoost.xlsx",
)


XGBoost best lambda: 50
XGBoost best run: 1
XGBoost ACP alpha_hat: 0.3400
XGBoost ACP width: 16.8823
XGBoost ACP coverage: 97.89%
XGBoost ACP RMSE: 3.9280
XGBoost ACP MAE: 2.7957
XGBoost ACP target coverage (1 - alpha~): 66.00%  |  miss / alpha~: 0.062  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         14.912430       0.754899         0.245101       0.726316  7.420220         0.362544
   20.0                   2         21.182261       0.533095         0.466905       0.778947 10.595130         0.414659
   50.0                   2         33.781052       0.340041         0.659959       0.978947 16.882284         0.061912


{'model': XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.7, colsample_bynode=None, colsample_bytree=0.7,
              device=None, early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=1, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.15, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=1, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=-1,
              num_parallel_tree=None, ...),
 'all_results': {10.0: {'all_losses': array([[15.545876, 15.533617, 15.488791, ..., 14.977953, 14.964459,
           14.973092],
          [15.479106, 15.531167, 15.493464, ..., 14.917636, 14.97517 ,
           14.91243 ],
          [15.551755

# **Conformal Predictions with GPBoost**


In [21]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')


In [22]:
run_full_acp_pipeline(
    GPBoostRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "GPBoost",
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/GPBoost.xlsx",
)


GPBoost best lambda: 50
GPBoost best run: 1
GPBoost ACP alpha_hat: 0.3372
GPBoost ACP width: 16.7538
GPBoost ACP coverage: 97.89%
GPBoost ACP RMSE: 3.5080
GPBoost ACP MAE: 2.6116
GPBoost ACP target coverage (1 - alpha~): 66.28%  |  miss / alpha~: 0.062  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         14.818499       0.748195         0.251805       0.747368  7.366076         0.337655
   20.0                   2         21.048634       0.525983         0.474017       0.873684 10.567536         0.240152
   50.0                   2         33.567905       0.337195         0.662805       0.978947 16.753841         0.062435


{'model': GPBoostRegressor(colsample_bytree=0.5, learning_rate=0.15, max_depth=5,
                  min_child_samples=1, min_child_weight=1e-05, n_estimators=300,
                  num_leaves=15, random_state=42, reg_alpha=0, reg_lambda=0,
                  subsample=0.6, verbose=-1),
 'all_results': {10.0: {'all_losses': array([[15.443838, 15.434628, 15.388993, ..., 14.865451, 14.8458  ,
           14.862695],
          [15.408787, 15.448107, 15.41667 , ..., 14.791028, 14.855968,
           14.818499],
          [15.455372, 15.373791, 15.409801, ..., 14.859479, 14.851479,
           14.852493]], dtype=float32),
   'all_sizes': array([[5.518123 , 5.561248 , 5.6155095, ..., 7.3719373, 7.318263 ,
           7.3852353],
          [5.482359 , 5.5537405, 5.5754857, ..., 7.2738085, 7.420871 ,
           7.2614317],
          [5.5329266, 5.4898505, 5.5959973, ..., 7.4198723, 7.2967935,
           7.377024 ]], dtype=float32),
   'all_alphas': array([[0.9925715 , 0.987338  , 0.9773484 , ..., 0.

# **Conformal Predictions with NGBoost**


In [23]:
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')


In [24]:
run_full_acp_pipeline(
    NGBRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "NGBoost",
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/NGBoost.xlsx",
)


NGBoost best lambda: 50
NGBoost best run: 1
NGBoost ACP alpha_hat: 0.3387
NGBoost ACP width: 16.8000
NGBoost ACP coverage: 96.84%
NGBoost ACP RMSE: 3.7253
NGBoost ACP MAE: 2.7502
NGBoost ACP target coverage (1 - alpha~): 66.13%  |  miss / alpha~: 0.093  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         14.857010       0.751751         0.248249       0.694737  7.385867         0.406070
   20.0                   2         21.104549       0.530702         0.469298       0.842105 10.549920         0.297521
   50.0                   2         33.660732       0.338737         0.661263       0.968421 16.799999         0.093226


{'model': NGBRegressor(learning_rate=0.03, minibatch_frac=0.5, n_estimators=1000,
              random_state=RandomState(MT19937) at 0x79840879AA40, verbose=0),
 'all_results': {10.0: {'all_losses': array([[15.491686 , 15.481385 , 15.435502 , ..., 14.921439 , 14.910182 ,
           14.917103 ],
          [15.436023 , 15.4800205, 15.454826 , ..., 14.838817 , 14.921653 ,
           14.85701  ],
          [15.504815 , 15.401307 , 15.456413 , ..., 14.906908 , 14.916777 ,
           14.914461 ]], dtype=float32),
   'all_sizes': array([[5.565983 , 5.608049 , 5.6621575, ..., 7.410661 , 7.334328 ,
           7.421649 ],
          [5.509575 , 5.5855985, 5.6135845, ..., 7.3016357, 7.4504347,
           7.2786007],
          [5.5823917, 5.517234 , 5.6427226, ..., 7.4366736, 7.3396945,
           7.407408 ]], dtype=float32),
   'all_alphas': array([[0.9925703 , 0.98733366, 0.97733444, ..., 0.7510779 , 0.7575854 ,
           0.7495454 ],
          [0.9926448 , 0.9894423 , 0.98412424, ..., 0.753718 

# **Conformal Predictions with Gradient Boosting**


In [25]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


In [26]:
run_full_acp_pipeline(
    GradientBoostingRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "GradientBoosting",
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/Gradient Boosting.xlsx",
)


GradientBoosting best lambda: 50
GradientBoosting best run: 1
GradientBoosting ACP alpha_hat: 0.3873
GradientBoosting ACP width: 19.4051
GradientBoosting ACP coverage: 95.79%
GradientBoosting ACP RMSE: 4.5169
GradientBoosting ACP MAE: 3.4482
GradientBoosting ACP target coverage (1 - alpha~): 61.27%  |  miss / alpha~: 0.109  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         17.109453       0.865973         0.134027       0.684211  8.494088         0.364664
   20.0                   2         24.288515       0.609225         0.390775       0.831579 12.162324         0.276451
   50.0                   2         38.692280       0.387327         0.612673       0.957895 19.405081         0.108707


{'model': GradientBoostingRegressor(alpha=0.5, learning_rate=0.2, loss='huber',
                           max_depth=5, max_leaf_nodes=10,
                           min_impurity_decrease=0.1, min_samples_split=10,
                           n_estimators=300, random_state=42, subsample=0.9),
 'all_results': {10.0: {'all_losses': array([[17.31206 , 17.319302, 17.297363, ..., 17.176811, 17.168064,
           17.140915],
          [17.284075, 17.299982, 17.299015, ..., 17.102613, 17.175184,
           17.109453],
          [17.32886 , 17.25125 , 17.310259, ..., 17.150717, 17.163912,
           17.145008]], dtype=float32),
   'all_sizes': array([[7.386346 , 7.4459596, 7.5244   , ..., 8.613642 , 8.392712 ,
           8.501106 ],
          [7.357671 , 7.405496 , 7.457805 , ..., 8.462045 , 8.571292 ,
           8.402309 ],
          [7.4064174, 7.3674836, 7.496613 , ..., 8.512523 , 8.508268 ,
           8.481857 ]], dtype=float32),
   'all_alphas': array([[0.9925714 , 0.98733425, 0.9772963 , 

# **Conformal Predictions with CatBoost**


In [27]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')


In [28]:
run_full_acp_pipeline(
    CatBoostRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "CatBoost",
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/CatBoost.xlsx",
)


CatBoost best lambda: 50
CatBoost best run: 1
CatBoost ACP alpha_hat: 0.3306
CatBoost ACP width: 16.7267
CatBoost ACP coverage: 95.79%
CatBoost ACP RMSE: 3.9613
CatBoost ACP MAE: 2.8948
CatBoost ACP target coverage (1 - alpha~): 66.94%  |  miss / alpha~: 0.127  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         14.637331       0.738091         0.261909       0.726316  7.303973         0.370800
   20.0                   2         20.793549       0.521627         0.478373       0.778947 10.422898         0.423775
   50.0                   2         33.164230       0.330583         0.669417       0.957895 16.726696         0.127367


{'model': <catboost.core.CatBoostRegressor at 0x7983ec3f8690>,
 'all_results': {10.0: {'all_losses': array([[15.326868, 15.314249, 15.272855, ..., 14.700511, 14.685043,
           14.696315],
          [15.287179, 15.3108  , 15.283039, ..., 14.643471, 14.701419,
           14.637331],
          [15.333553, 15.257314, 15.290185, ..., 14.703305, 14.700002,
           14.701877]], dtype=float32),
   'all_sizes': array([[5.4011607, 5.440886 , 5.499634 , ..., 7.299854 , 7.2203016,
           7.314862 ],
          [5.3607516, 5.4163475, 5.4416914, ..., 7.2249546, 7.3183904,
           7.180207 ],
          [5.411113 , 5.373428 , 5.4764824, ..., 7.329203 , 7.2384863,
           7.286729 ]], dtype=float32),
   'all_alphas': array([[0.9925707 , 0.9873362 , 0.9773221 , ..., 0.74006575, 0.74647415,
           0.7381453 ],
          [0.99264276, 0.98944515, 0.9841349 , ..., 0.7418517 , 0.73830277,
           0.7457124 ],
          [0.99224406, 0.98838866, 0.98137015, ..., 0.73741025, 0.7461516 ,
 

# **Conformal Predictions with HistGradientBoosting**


In [29]:
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')


In [30]:
run_full_acp_pipeline(
    HistGradientBoostingRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "HGBR",
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/HGBM.xlsx",
)


HGBR best lambda: 50
HGBR best run: 1
HGBR ACP alpha_hat: 0.3564
HGBR ACP width: 17.5112
HGBR ACP coverage: 95.79%
HGBR ACP RMSE: 4.2407
HGBR ACP MAE: 3.1603
HGBR ACP target coverage (1 - alpha~): 64.36%  |  miss / alpha~: 0.118  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         15.577388       0.789612         0.210388       0.684211  7.721207         0.399930
   20.0                   2         22.122057       0.556057         0.443943       0.778947 11.052370         0.397536
   50.0                   2         35.266613       0.356385         0.643615       0.957895 17.511217         0.118146


{'model': HistGradientBoostingRegressor(early_stopping=False, l2_regularization=0.1,
                               max_bins=64, max_depth=7, max_leaf_nodes=63,
                               min_samples_leaf=5, random_state=42),
 'all_results': {10.0: {'all_losses': array([[16.04692 , 16.043818, 16.004137, ..., 15.647894, 15.612734,
           15.621948],
          [15.97966 , 16.033218, 16.017797, ..., 15.537825, 15.628954,
           15.577388],
          [16.057682, 15.958928, 16.017542, ..., 15.632727, 15.630447,
           15.633763]], dtype=float32),
   'all_sizes': array([[6.1212254, 6.170553 , 6.231106 , ..., 7.7910085, 7.642645 ,
           7.803997 ],
          [6.053208 , 6.1387906, 6.176645 , ..., 7.651479 , 7.803639 ,
           7.648041 ],
          [6.1352587, 6.074957 , 6.203911 , ..., 7.78787  , 7.700632 ,
           7.7696443]], dtype=float32),
   'all_alphas': array([[0.9925695 , 0.9873264 , 0.9773031 , ..., 0.78568846, 0.79700893,
           0.78179514],
          

# **Conformal Predictions with PGBM**


In [31]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [32]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())


Training on CPU


Estimator 0/300, Train metric: 9.3540
Estimator 1/300, Train metric: 8.7014
Estimator 2/300, Train metric: 8.2176
Estimator 3/300, Train metric: 7.7928
Estimator 4/300, Train metric: 7.4588
Estimator 5/300, Train metric: 7.0766
Estimator 6/300, Train metric: 6.7824
Estimator 7/300, Train metric: 6.3936
Estimator 8/300, Train metric: 6.0929
Estimator 9/300, Train metric: 5.8563
Estimator 10/300, Train metric: 5.5902


Estimator 11/300, Train metric: 5.3150
Estimator 12/300, Train metric: 5.1076
Estimator 13/300, Train metric: 4.9502
Estimator 14/300, Train metric: 4.7788
Estimator 15/300, Train metric: 4.6107
Estimator 16/300, Train metric: 4.4217
Estimator 17/300, Train metric: 4.2602
Estimator 18/300, Train metric: 4.0980
Estimator 19/300, Train metric: 3.9577
Estimator 20/300, Train metric: 3.8037
Estimator 21/300, Train metric: 3.6834
Estimator 22/300, Train metric: 3.5716
Estimator 23/300, Train metric: 3.4367
Estimator 24/300, Train metric: 3.3685
Estimator 25/300, Train metric: 3.2836
Estimator 26/300, Train metric: 3.1784
Estimator 27/300, Train metric: 3.1301


Estimator 28/300, Train metric: 3.0346
Estimator 29/300, Train metric: 2.9881
Estimator 30/300, Train metric: 2.9410
Estimator 31/300, Train metric: 2.8818
Estimator 32/300, Train metric: 2.8382
Estimator 33/300, Train metric: 2.7729
Estimator 34/300, Train metric: 2.7350
Estimator 35/300, Train metric: 2.6757
Estimator 36/300, Train metric: 2.6177
Estimator 37/300, Train metric: 2.5424
Estimator 38/300, Train metric: 2.5075
Estimator 39/300, Train metric: 2.4696
Estimator 40/300, Train metric: 2.4056
Estimator 41/300, Train metric: 2.3744
Estimator 42/300, Train metric: 2.3309
Estimator 43/300, Train metric: 2.2890
Estimator 44/300, Train metric: 2.2685
Estimator 45/300, Train metric: 2.2192


Estimator 46/300, Train metric: 2.2087
Estimator 47/300, Train metric: 2.1725
Estimator 48/300, Train metric: 2.1472
Estimator 49/300, Train metric: 2.1308
Estimator 50/300, Train metric: 2.0869
Estimator 51/300, Train metric: 2.0626
Estimator 52/300, Train metric: 2.0371
Estimator 53/300, Train metric: 2.0172
Estimator 54/300, Train metric: 1.9750
Estimator 55/300, Train metric: 1.9376
Estimator 56/300, Train metric: 1.9247
Estimator 57/300, Train metric: 1.8948
Estimator 58/300, Train metric: 1.8667
Estimator 59/300, Train metric: 1.8407
Estimator 60/300, Train metric: 1.8067
Estimator 61/300, Train metric: 1.7990
Estimator 62/300, Train metric: 1.7828
Estimator 63/300, Train metric: 1.7553


Estimator 64/300, Train metric: 1.7234
Estimator 65/300, Train metric: 1.7172
Estimator 66/300, Train metric: 1.6990
Estimator 67/300, Train metric: 1.6698
Estimator 68/300, Train metric: 1.6585
Estimator 69/300, Train metric: 1.6427
Estimator 70/300, Train metric: 1.6199
Estimator 71/300, Train metric: 1.6148
Estimator 72/300, Train metric: 1.5931
Estimator 73/300, Train metric: 1.5767
Estimator 74/300, Train metric: 1.5664
Estimator 75/300, Train metric: 1.5543
Estimator 76/300, Train metric: 1.5420
Estimator 77/300, Train metric: 1.5259
Estimator 78/300, Train metric: 1.5041
Estimator 79/300, Train metric: 1.4966
Estimator 80/300, Train metric: 1.4889
Estimator 81/300, Train metric: 1.4770


... (10 outputs left out)

Estimator 254/300, Train metric: 0.6958
Estimator 255/300, Train metric: 0.6950
Estimator 256/300, Train metric: 0.6937
Estimator 257/300, Train metric: 0.6924
Estimator 258/300, Train metric: 0.6910
Estimator 259/300, Train metric: 0.6915
Estimator 260/300, Train metric: 0.6902
Estimator 261/300, Train metric: 0.6865
Estimator 262/300, Train metric: 0.6858
Estimator 263/300, Train metric: 0.6851
Estimator 264/300, Train metric: 0.6847
Estimator 265/300, Train metric: 0.6830
Estimator 266/300, Train metric: 0.6815
Estimator 267/300, Train metric: 0.6809
Estimator 268/300, Train metric: 0.6813
Estimator 269/300, Train metric: 0.6801
Estimator 270/300, Train metric: 0.6788


Estimator 271/300, Train metric: 0.6774
Estimator 272/300, Train metric: 0.6748
Estimator 273/300, Train metric: 0.6747
Estimator 274/300, Train metric: 0.6742
Estimator 275/300, Train metric: 0.6732
Estimator 276/300, Train metric: 0.6701
Estimator 277/300, Train metric: 0.6680
Estimator 278/300, Train metric: 0.6676
Estimator 279/300, Train metric: 0.6642
Estimator 280/300, Train metric: 0.6630
Estimator 281/300, Train metric: 0.6630
Estimator 282/300, Train metric: 0.6619
Estimator 283/300, Train metric: 0.6608
Estimator 284/300, Train metric: 0.6602
Estimator 285/300, Train metric: 0.6581
Estimator 286/300, Train metric: 0.6562
Estimator 287/300, Train metric: 0.6562
Estimator 288/300, Train metric: 0.6535
Estimator 289/300, Train metric: 0.6510


Estimator 290/300, Train metric: 0.6493
Estimator 291/300, Train metric: 0.6458
Estimator 292/300, Train metric: 0.6446
Estimator 293/300, Train metric: 0.6449
Estimator 294/300, Train metric: 0.6438
Estimator 295/300, Train metric: 0.6426
Estimator 296/300, Train metric: 0.6412
Estimator 297/300, Train metric: 0.6405
Estimator 298/300, Train metric: 0.6394
Estimator 299/300, Train metric: 0.6384


        0.05        0.1       0.15        0.2        0.3        0.4  \
0  32.613903  32.712513  32.806610  32.871300  32.933006  32.968895   
1  36.357594  36.489403  36.528233  36.565876  36.623905  36.670753   
2  34.306561  34.349525  34.520794  34.604797  34.688782  34.740742   
3  36.493179  36.536526  36.618469  36.665985  36.766624  36.820503   
4  42.051926  42.244644  42.344818  42.475914  42.562881  42.680779   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0  33.013924  33.078548  33.141739  33.208607  33.240299  33.328312   
1  36.722866  36.785667  36.856281  36.925179  36.957905  37.133049   
2  34.778641  34.845863  34.908909  35.034164  35.110901  35.156506   
3  36.903473  36.965099  37.034515  37.142761  37.214592  37.252678   
4  42.777512  42.944244  43.104347  43.261200  43.377827  43.567608   

        0.95  Actual  
0  33.510792    34.0  
1  37.218525    36.0  
2  35.293854    36.0  
3  37.407856    35.0  
4  43.795647    44.6  


In [33]:
# ACP execution for PGBM
best_params = get_best_model_params(best_scores_autosampler, "PGBM")
run_full_acp_pipeline(
    model_class=PGBMWrapper,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/PGBM.xlsx",
)


Training on CPU
Estimator 0/300, Train metric: 9.8639
Estimator 1/300, Train metric: 9.4198
Estimator 2/300, Train metric: 8.8706
Estimator 3/300, Train metric: 8.2754
Estimator 4/300, Train metric: 7.9336
Estimator 5/300, Train metric: 7.4593
Estimator 6/300, Train metric: 7.1351
Estimator 7/300, Train metric: 6.8551
Estimator 8/300, Train metric: 6.5315
Estimator 9/300, Train metric: 6.1990
Estimator 10/300, Train metric: 5.9212
Estimator 11/300, Train metric: 5.6747
Estimator 12/300, Train metric: 5.4636
Estimator 13/300, Train metric: 5.2518
Estimator 14/300, Train metric: 5.0380
Estimator 15/300, Train metric: 4.9005


Estimator 16/300, Train metric: 4.7332
Estimator 17/300, Train metric: 4.5405
Estimator 18/300, Train metric: 4.4189


Estimator 19/300, Train metric: 4.2608
Estimator 20/300, Train metric: 4.1335
Estimator 21/300, Train metric: 4.0506
Estimator 22/300, Train metric: 3.9283
Estimator 23/300, Train metric: 3.8329
Estimator 24/300, Train metric: 3.7099
Estimator 25/300, Train metric: 3.6085
Estimator 26/300, Train metric: 3.4763
Estimator 27/300, Train metric: 3.3771
Estimator 28/300, Train metric: 3.3030
Estimator 29/300, Train metric: 3.2219
Estimator 30/300, Train metric: 3.1719
Estimator 31/300, Train metric: 3.0861
Estimator 32/300, Train metric: 3.0474
Estimator 33/300, Train metric: 2.9513
Estimator 34/300, Train metric: 2.9318
Estimator 35/300, Train metric: 2.8883


Estimator 36/300, Train metric: 2.8419
Estimator 37/300, Train metric: 2.8288
Estimator 38/300, Train metric: 2.7943


Estimator 39/300, Train metric: 2.7551
Estimator 40/300, Train metric: 2.7129
Estimator 41/300, Train metric: 2.6913
Estimator 42/300, Train metric: 2.6704
Estimator 43/300, Train metric: 2.6400
Estimator 44/300, Train metric: 2.6021
Estimator 45/300, Train metric: 2.5911
Estimator 46/300, Train metric: 2.5335
Estimator 47/300, Train metric: 2.5151
Estimator 48/300, Train metric: 2.4743
Estimator 49/300, Train metric: 2.4302
Estimator 50/300, Train metric: 2.4034
Estimator 51/300, Train metric: 2.3593
Estimator 52/300, Train metric: 2.3313
Estimator 53/300, Train metric: 2.2874
Estimator 54/300, Train metric: 2.2604
Estimator 55/300, Train metric: 2.2240


Estimator 56/300, Train metric: 2.1930
Estimator 57/300, Train metric: 2.1787
Estimator 58/300, Train metric: 2.1485


... (22 outputs left out)

Estimator 297/300, Train metric: 0.6798
Estimator 298/300, Train metric: 0.6782
Estimator 299/300, Train metric: 0.6755


PGBM best lambda: 50
PGBM best run: 1
PGBM ACP alpha_hat: 0.3392
PGBM ACP width: 17.0167
PGBM ACP coverage: 97.89%
PGBM ACP RMSE: 3.6157
PGBM ACP MAE: 2.6409
PGBM ACP target coverage (1 - alpha~): 66.08%  |  miss / alpha~: 0.062  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         14.968700       0.755980         0.244020       0.715789  7.449654         0.375950
   20.0                   2         21.261761       0.532998         0.467002       0.852632 10.654690         0.276490
   50.0                   2         33.906822       0.339225         0.660775       0.978947 17.016670         0.062061


{'model': PGBMWrapper(),
 'all_results': {10.0: {'all_losses': array([[15.560521 , 15.56217  , 15.524106 , ..., 15.02099  , 15.01317  ,
           15.008703 ],
          [15.518534 , 15.55029  , 15.5365925, ..., 14.943835 , 15.025031 ,
           14.9687   ],
          [15.5776415, 15.499665 , 15.535344 , ..., 15.011094 , 15.023905 ,
           15.011363 ]], dtype=float32),
   'all_sizes': array([[5.6347947, 5.6888375, 5.751036 , ..., 7.449094 , 7.4062204,
           7.4415483],
          [5.592089 , 5.6558094, 5.6953588, ..., 7.3676934, 7.4961424,
           7.33788  ],
          [5.6551943, 5.615781 , 5.7216086, ..., 7.4741287, 7.4105377,
           7.4333997]], dtype=float32),
   'all_alphas': array([[0.99257267, 0.98733324, 0.9773069 , ..., 0.75718975, 0.7606951 ,
           0.7567155 ],
          [0.9926445 , 0.9894481 , 0.98412347, ..., 0.75761414, 0.75288886,
           0.763082  ],
          [0.9922448 , 0.9883884 , 0.9813735 , ..., 0.75369644, 0.76133674,
           0.75779635

# **Conformal Predictions with TabNet**


In [34]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')
print(best_params)


{'n_d': 32, 'n_a': 32, 'n_steps': 3, 'gamma': 1.5, 'lambda_sparse': 0.0001, 'optimizer_params': {'lr': 0.02}, 'mask_type': 'entmax', 'n_shared': 1, 'n_independent': 1, 'scheduler_params': {'step_size': 10, 'gamma': 0.9}, 'scheduler_fn': <class 'torch.optim.lr_scheduler.StepLR'>, 'seed': 42, 'verbose': 0}


In [35]:
class TabNetRegressorCP(TabNetRegressor):
    def fit(self, X, y, *args, **kwargs):
        y_arr = np.asarray(y)
        if y_arr.ndim == 1:
            y_arr = y_arr.reshape(-1, 1)
        return super().fit(X, y_arr, *args, **kwargs)

    def predict(self, X, *args, **kwargs):
        preds = super().predict(X, *args, **kwargs)
        return np.asarray(preds).reshape(-1)


best_params = get_best_model_params(best_scores_autosampler, "TabNet")
if isinstance(best_params, dict) and "verbose" in best_params:
    best_params = dict(best_params)
    best_params.pop("verbose", None)

run_full_acp_pipeline(
    TabNetRegressorCP,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "TabNet",
    "./drive/MyDrive/Regression/concrete_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/TabNet.xlsx",
)


epoch 0  | loss: 1273.13586|  0:00:00s
epoch 1  | loss: 1173.28699|  0:00:00s
epoch 2  | loss: 1063.66394|  0:00:00s
epoch 3  | loss: 966.99445|  0:00:00s
epoch 4  | loss: 878.01044|  0:00:00s


epoch 5  | loss: 799.25183|  0:00:00s
epoch 6  | loss: 726.86542|  0:00:00s
epoch 7  | loss: 652.49298|  0:00:00s
epoch 8  | loss: 581.2818|  0:00:00s
epoch 9  | loss: 512.27728|  0:00:00s
epoch 10 | loss: 446.71402|  0:00:00s
epoch 11 | loss: 382.29797|  0:00:00s


epoch 12 | loss: 321.33218|  0:00:00s
epoch 13 | loss: 264.78964|  0:00:00s
epoch 14 | loss: 213.00606|  0:00:00s
epoch 15 | loss: 165.98282|  0:00:00s
epoch 16 | loss: 126.02397|  0:00:00s


epoch 17 | loss: 91.43346|  0:00:00s
epoch 18 | loss: 64.81685|  0:00:00s
epoch 19 | loss: 47.30199|  0:00:00s
epoch 20 | loss: 37.86909|  0:00:00s
epoch 21 | loss: 38.32196|  0:00:00s
epoch 22 | loss: 38.69481|  0:00:00s
epoch 23 | loss: 45.04677|  0:00:00s
epoch 24 | loss: 51.03404|  0:00:00s


epoch 25 | loss: 54.08438|  0:00:00s
epoch 26 | loss: 49.10545|  0:00:00s
epoch 27 | loss: 44.75362|  0:00:00s
epoch 28 | loss: 33.90328|  0:00:00s
epoch 29 | loss: 26.43323|  0:00:00s


epoch 30 | loss: 25.28057|  0:00:00s
epoch 31 | loss: 23.93259|  0:00:00s
epoch 32 | loss: 21.48399|  0:00:00s
epoch 33 | loss: 19.77467|  0:00:00s
epoch 34 | loss: 20.92794|  0:00:00s
epoch 35 | loss: 19.8964 |  0:00:00s
epoch 36 | loss: 17.9838 |  0:00:00s
epoch 37 | loss: 14.71654|  0:00:00s


... (9 outputs left out)

epoch 94 | loss: 2.5592  |  0:00:01s
epoch 95 | loss: 2.43541 |  0:00:01s
epoch 96 | loss: 2.42493 |  0:00:01s
epoch 97 | loss: 2.49631 |  0:00:01s
epoch 98 | loss: 2.65527 |  0:00:01s
epoch 99 | loss: 2.20681 |  0:00:01s


TabNet best lambda: 50
TabNet best run: 1
TabNet ACP alpha_hat: 0.5454
TabNet ACP width: 27.6947
TabNet ACP coverage: 84.21%
TabNet ACP RMSE: 10.0844
TabNet ACP MAE: 6.9439
TabNet ACP target coverage (1 - alpha~): 45.46%  |  miss / alpha~: 0.289  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage     width  Miss_over_alpha
   10.0                   2         24.778353       0.999000         0.001000       0.726316 14.932774         0.273958
   20.0                   2         34.419254       0.870479         0.129521       0.747368 17.175392         0.290221
   50.0                   2         54.716019       0.545431         0.454569       0.842105 27.694651         0.289486


{'model': TabNetRegressorCP(n_d=32, n_a=32, n_steps=3, gamma=1.5, cat_idxs=[], cat_dims=[], cat_emb_dim=[], n_independent=1, n_shared=1, epsilon=1e-15, momentum=0.02, lambda_sparse=0.0001, seed=42, clip_value=1, verbose=1, optimizer_fn=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.02}, scheduler_fn=<class 'torch.optim.lr_scheduler.StepLR'>, scheduler_params={'step_size': 10, 'gamma': 0.9}, mask_type='entmax', input_dim=7, output_dim=1, device_name='auto', n_shared_decoder=1, n_indep_decoder=1, grouped_features=[], compile_backend=''),
 'all_results': {10.0: {'all_losses': array([[24.925884, 24.951593, 24.936436, ..., 24.942541, 24.942812,
           24.942003],
          [24.886805, 24.913094, 24.928013, ..., 24.890514, 24.961864,
           24.778353],
          [24.974268, 24.790037, 24.951683, ..., 24.940735, 24.928783,
           24.97351 ]], dtype=float32),
   'all_sizes': array([[14.978424 , 14.982157 , 14.953855 , ..., 14.952541 , 14.95281  ,
           14.952004 ],